# Hands-On Grounding — Qdrant en mémoire

[← RAG et Mémoire Sémantique](README.md) · [↑ GenAI](../README.md) · [Suite : retrieval avancé →](02-Retrieval-Avance.ipynb)

Ce notebook est le **versant pratique** de la section [RAG et Mémoire Sémantique](README.md).
Il fait tourner une vraie base vectorielle **Qdrant en mémoire** (`:memory:`), sans Docker,
sans GPU, sans clé d'API et sans aucune donnée réelle. Tout est reproductible en local.

Vous y manipulerez, étape par étape, les quatre gestes qui forment le socle du *grounding* :

1. **Embarquer** un petit corpus (texte → vecteurs) ;
2. **Rechercher par le sens** (et voir où le `grep` échoue) ;
3. **Filtrer** sur la charge utile (*payload*) ;
4. **Indexer** un champ de payload — et reproduire en miniature l'incident réel
   du « *scroll doom loop* » (cf. [04 - Incidents et leçons](docs/04-Incidents-et-Lecons.md), section 8).

### Carte courte de la série

Le parcours se lit comme une chaîne de responsabilités :

1. **Grounding** — ce notebook pose le corpus, les vecteurs et les filtres.
2. **Retrieval** — [`02-Retrieval-Avance.ipynb`](02-Retrieval-Avance.ipynb) classe et combine les résultats.
3. **Fondations from scratch** — [`03-Embeddings-From-Scratch.ipynb`](03-Embeddings-From-Scratch.ipynb) construit la géométrie ; [`04-Tokenisation-From-Scratch.ipynb`](04-Tokenisation-From-Scratch.ipynb) construit l'unité de compte.
4. **Stockage** — [`05-Stockage-Vectoriel.ipynb`](05-Stockage-Vectoriel.ipynb) couvre le moteur local ; [`05b-Stockage-Vectoriel-Serveur.ipynb`](05b-Stockage-Vectoriel-Serveur.ipynb) passe au serveur.
5. **KernelMemory** — [`06`](06-KernelMemory-InProcess.ipynb) introduit le pipeline, [`07`](07-KernelMemory-Python-Quickstart.ipynb) le transpose en Python, puis [`08`](08-KernelMemory-Hybrid-Search.ipynb) et [`09`](09-KernelMemory-Multimodal.ipynb) étendent la recherche hybride et multimodale.

> Prérequis conceptuels : [01 - Pourquoi la mémoire sémantique](docs/01-Pourquoi-Memoire-Semantique.md).
> Aucune installation lourde : `qdrant-client` embarque un moteur en mémoire et `fastembed`
> télécharge un petit modèle d'embeddings au premier appel (la taille réellement
> téléchargée se lit dans le commentaire de la section 2, mesurée sur le cache local).

## 1. Installation

`qdrant-client[fastembed]` apporte à la fois le client Qdrant et l'intégration `fastembed`
(modèle d'embeddings local, exécuté sur CPU). Aucune clé d'API n'est requise.

In [1]:
# Décommentez pour installer (première exécution) :
# %pip install -q "qdrant-client[fastembed]>=1.20"

from qdrant_client import QdrantClient, models
print("qdrant-client importé — moteur en mémoire prêt.")

qdrant-client importé — moteur en mémoire prêt.


### Lecture du résultat : import sans clé d'API

L'output ci-dessus affiche *"qdrant-client importé — moteur en mémoire prêt"*, ce qui confirme **trois choses en un seul geste** : (1) le paquet `qdrant-client[fastembed]` a été installé dans l'environnement actif sans erreur de résolution de dépendances, (2) le moteur Qdrant **embarqué** (mode `:memory:`) est prêt à recevoir des opérations — sans qu'un serveur séparé ne tourne à côté, (3) aucun appel réseau ni clé d'API n'a été nécessaire pour atteindre cet état. C'est la **promesse du mode mémoire** : reproduire en local la même API qu'une instance Docker de production, sans l'infrastructure. Si l'import échoue, le message d'erreur vient de Python (paquet absent, version incompatible) — il faut alors **installer le paquet** plutôt que chercher à contourner (cf règle F : on répare l'environnement, on ne maquille pas l'erreur).

## 2. Une base vectorielle en mémoire

`QdrantClient(":memory:")` instancie un moteur Qdrant **dans le processus Python**, sans
serveur ni conteneur. L'API est **identique** à celle d'un Qdrant de production : tout ce
que vous écrivez ici se transpose tel quel vers une instance Docker (cf. [02 - Infrastructure](docs/02-Infrastructure-Qdrant.md)).

In [2]:
import os
# Cache huggingface_hub : Windows ne supporte pas les symlinks du cache modeles ; bascule
# amont documentee (bruit machine, sans effet sur le telechargement ni les vecteurs).
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

client = QdrantClient(":memory:")

# Modèle d'embeddings MULTILINGUE (gère le français), exécuté localement sur CPU via
# fastembed. 384 dimensions, ~0,22 Go téléchargé au premier appel. On l'explicite pour
# la reproductibilité — un modèle anglais seul séparerait mal des phrases françaises.
# Depuis qdrant-client 1.17, le client ne porte plus de modèle par défaut : le modèle se
# déclare sur chaque document (`models.Document`) et la collection annonce sa dimension.
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM = 384

client.create_collection(
    "grounding_demo",
    vectors_config=models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE),
)
print(
    "Modèle d'embeddings :",
    EMBED_MODEL,
    f"— collection « grounding_demo » créée ({EMBED_DIM} dimensions).",
)

Modèle d'embeddings : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 — collection « grounding_demo » créée (384 dimensions).


### Lecture du résultat : choix du modèle multilingue

La ligne *"Modèle d'embeddings : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 — collection « grounding_demo » créée (384 dimensions)"* confirme deux choses en un seul geste : le modèle d'embeddings **multilingue** est résolu, et la collection est ouverte avec la dimension attendue (384). Le choix du modèle n'est pas anodin : ce modèle rapproche correctement des phrases françaises **entre elles** (par exemple *"éviter les pics de charge"* et *"throttling des appels sortants"*), là où un modèle purement anglais sous-représenterait leur parent sémantique. **Si le corpus contient surtout du français, garder ce modèle** ; pour un corpus anglais technique dense, un modèle BGE ou E5 peut donner de meilleurs résultats — c'est précisément la variation que l'exemple guidé 3 explore. Le premier appel télécharge le modèle puis le met en cache (la taille constatée sur le cache local est indiquée dans le commentaire de la section 2) ; les appels suivants sont servis depuis ce cache par `fastembed`.

Depuis `qdrant-client` 1.20, le modèle ne se règle plus **une fois pour toutes** sur le client : `set_model` et `add` ont été retirés, et le modèle se déclare **à chaque document**, dans `models.Document(text=..., model=...)`. C'est plus verbeux, mais c'est ce qui permet à deux collections de cohabiter avec deux modèles différents dans le même processus — c'est exactement ce que fait l'exemple guidé 3.

## 3. Indexer un petit corpus

On indexe une poignée de fragments imitant ce qu'une flotte d'agents produit : des bribes
de conversations et de notes techniques. À chaque document est attachée une **charge utile**
(*payload*) — ici la `source`, le `sujet` et le `texte` lui-même — qui voyagera avec le
vecteur et servira aux filtres.

On construit un `models.PointStruct` par document : `vector=models.Document(text=..., model=...)`
demande à `fastembed` d'embarquer le texte, `payload=...` transporte les métadonnées. Un
`client.upsert(...)` écrit ensuite les points dans la collection.

In [3]:
documents = [
    "Pour éviter de saturer l'API, on limite le nombre de requêtes par seconde.",
    "Le throttling des appels sortants protège le service des pics de charge.",
    "La collection a été recréée après une dérive de montage du disque virtuel.",
    "Un index de payload sur le champ de hash rend la déduplication instantanée.",
    "La quantization TurboQuant 4 bits comprime les vecteurs d'un facteur huit.",
    "La recette de la tarte aux pommes demande des pommes, du beurre et du sucre.",
]

metadata = [
    {"source": "conversation", "sujet": "rate-limiting"},
    {"source": "code",         "sujet": "rate-limiting"},
    {"source": "incident",     "sujet": "mount-drift"},
    {"source": "incident",     "sujet": "payload-index"},
    {"source": "doc",          "sujet": "quantization"},
    {"source": "bruit",        "sujet": "hors-domaine"},
]
ids = list(range(len(documents)))

# Le texte est conservé dans le payload : depuis qdrant-client 1.17, un point ne porte
# plus le document d'origine à côté de son vecteur — c'est le payload qui le restitue.
points = [
    models.PointStruct(
        id=ids[i],
        vector=models.Document(text=texte, model=EMBED_MODEL),
        payload={**metadata[i], "texte": texte},
    )
    for i, texte in enumerate(documents)
]
client.upsert(collection_name="grounding_demo", points=points)
info = client.get_collection("grounding_demo")
print("Points indexés :", info.points_count)

Points indexés : 6


### Lecture du résultat : un corpus hétérogène pour un test réaliste

*"Points indexés : 6"* — le corpus se compose volontairement de **5 fragments métier** (rate-limiting, dérive de montage, index de payload, quantization) **et 1 fragment hors-domaine** (la tarte aux pommes). Cette dernière phrase n'a rien à faire dans une base de production — et c'est précisément ce qui rend la démo probante : on va voir en section 4 qu'elle n'apparaît même pas dans le top 3 d'une requête métier, ce qui **démontre que le moteur ne confond pas le sujet et le décor**. En production, ce genre de fragment *bruit* sert de **témoin négatif** : si une requête "innocente" remonte la tarte aux pommes dans le top 3, c'est que quelque chose a dérivé (modèle trop générique, corpus mal segmenté, ou `limit` trop élevé).

## 4. Recherche sémantique

On interroge en **langage naturel**. Qdrant embarque la requête avec le même modèle,
puis classe les points par similarité cosinus décroissante.
Pour deux vecteurs non nuls, cette similarité vaut :

$$s(q,d)=\frac{q\cdot d}{\lVert q\rVert\lVert d\rVert}\in[-1,1].$$

- Un score élevé indique des directions plus proches dans l'espace du modèle.
- Un score négatif est possible : l'exemple guidé 3 en montre.
- Le score n'est ni une probabilité ni une garantie de pertinence métier.

`limit=k` demande les k meilleurs points disponibles, sans imposer de score minimal.
Le classement exprime une proximité relative au corpus et au modèle utilisés.
Un seuil de rejet doit être évalué sur des requêtes et documents de pertinence connue ;
l'exemple guidé 3 illustre pourquoi le recalibrer lorsque le modèle change.

In [4]:
def chercher(question, limit=3):
    reponse = client.query_points(
        collection_name="grounding_demo",
        query=models.Document(text=question, model=EMBED_MODEL),
        limit=limit,
    )
    hits = reponse.points
    print(f"Requête : {question!r}\n")
    for h in hits:
        print(
            f"  score={h.score:.3f}  "
            f"[{h.payload['source']}/{h.payload['sujet']}]  {h.payload['texte']}"
        )
    print()
    return hits

_ = chercher("comment ne pas dépasser le quota d'appels ?")

Requête : "comment ne pas dépasser le quota d'appels ?"

  score=0.667  [code/rate-limiting]  Le throttling des appels sortants protège le service des pics de charge.
  score=0.519  [conversation/rate-limiting]  Pour éviter de saturer l'API, on limite le nombre de requêtes par seconde.
  score=0.113  [incident/payload-index]  Un index de payload sur le champ de hash rend la déduplication instantanée.



### Lecture du résultat : la tarte aux pommes n'apparaît pas dans le top 3

Trois scores, dans l'ordre : **0.667** (*[code/rate-limiting]*), **0.519** (*[conversation/rate-limiting]*), **0.113** (*[incident/payload-index]*). La tarte aux pommes (sujet `hors-domaine`) n'apparaît même pas dans ce top 3 — c'est ce qu'annonce la section 5, juste après : *"le document tarte aux pommes reste loin : il partage peut-être des mots fonctionnels, mais pas le sens"*. Le **0.113** du fragment payload-index est trompeur : il est bas (le document parle de hash, pas de throttling), mais **non nul** — c'est le reflet d'un voisinage sémantique large dans l'espace des embeddings, pas une proximité réelle. Une **règle métier** saine consiste à filtrer les hits en dessous d'un seuil de score (par exemple `score ≥ 0.3`) plutôt que d'espérer que le moteur classe tout par pertinence absolue.

## 5. Sémantique contre lexical : le `grep` est aveugle au sens

La requête ci-dessus ne contient **aucun** des mots « limite », « requêtes » ou « throttling ».
Un `grep` n'aurait rien trouvé. La recherche sémantique, elle, rapproche *« quota d'appels »*,
*« limiter les requêtes »* et *« throttling »* parce qu'ils **signifient** la même chose.

À l'inverse, le document « tarte aux pommes » reste loin : il partage peut-être des mots
fonctionnels, mais pas le sens.

In [5]:
# Trois formulations très différentes du même concept -> mêmes documents pertinents.
for q in ["rate limiting", "éviter les pics de charge sur le service", "ralentir les appels réseau"]:
    _ = chercher(q, limit=2)

Requête : 'rate limiting'

  score=0.349  [code/rate-limiting]  Le throttling des appels sortants protège le service des pics de charge.
  score=0.344  [conversation/rate-limiting]  Pour éviter de saturer l'API, on limite le nombre de requêtes par seconde.

Requête : 'éviter les pics de charge sur le service'

  score=0.714  [code/rate-limiting]  Le throttling des appels sortants protège le service des pics de charge.
  score=0.390  [conversation/rate-limiting]  Pour éviter de saturer l'API, on limite le nombre de requêtes par seconde.

Requête : 'ralentir les appels réseau'

  score=0.615  [code/rate-limiting]  Le throttling des appels sortants protège le service des pics de charge.
  score=0.428  [conversation/rate-limiting]  Pour éviter de saturer l'API, on limite le nombre de requêtes par seconde.



### Lecture du résultat : trois formulations, un même top document

Les trois requêtes affichées placent *code/rate-limiting* au premier rang.
Elles ne contiennent pas « throttling », terme présent dans le document retrouvé :
sur ce petit exemple, le rapprochement ne se limite donc pas à ce mot exact.

La sortie montre aussi que la formulation anglaise reçoit ici un score inférieur
aux deux formulations françaises. Ce constat ne suffit pas à attribuer l'écart
à la langue : les formulations changent aussi. Il faudrait des comparaisons
contrôlées pour isoler cet effet.

Ces scores décrivent ce corpus avec ce modèle ; ils ne calibrent pas un autre
encodeur. L'exemple guidé 3 compare les rangs et les échelles de deux modèles.

Le premier rang répond à une question de classement, pas à celle d'un seuil
d'acceptation : même le meilleur résultat peut être hors sujet. Pour choisir
un seuil, confrontez les scores à des exemples pertinents et non pertinents
de votre application, puis réévaluez-le si le corpus ou l'encodeur change.

## 6. Filtrer sur la charge utile (*payload*)

La recherche sémantique peut être **contrainte** par un filtre exact sur le payload : « les
fragments proches de cette idée, **mais seulement** ceux issus d'un incident ». C'est la
combinaison vecteur + filtre qui fait la puissance d'une base comme Qdrant.

In [6]:
flt = models.Filter(
    must=[models.FieldCondition(key="source", match=models.MatchValue(value="incident"))]
)
reponse = client.query_points(
    collection_name="grounding_demo",
    query=models.Document(text="problème d'infrastructure à diagnostiquer", model=EMBED_MODEL),
    query_filter=flt,
    limit=3,
)
for h in reponse.points:
    print(
        f"  score={h.score:.3f}  "
        f"[{h.payload['source']}/{h.payload['sujet']}]  {h.payload['texte']}"
    )

  score=0.276  [incident/payload-index]  Un index de payload sur le champ de hash rend la déduplication instantanée.
  score=0.116  [incident/mount-drift]  La collection a été recréée après une dérive de montage du disque virtuel.


### Lecture du résultat : le filtre payload borne la recherche au bon périmètre

Deux hits seulement après filtre `source = 'incident'` : **0.276** (*[incident/payload-index]*) et **0.116** (*[incident/mount-drift]*). Les fragments *rate-limiting* (qui avaient les meilleurs scores à la cellule 8) sont **exclus**, même s'ils sont sémantiquement plus proches — parce qu'ils n'ont pas la bonne valeur de payload. C'est la **combinaison des deux signaux** (vecteur + métadonnée) qui fait la puissance d'une base hybride : la sémantique répond à *"qu'est-ce qui parle de ce sujet ?"*, le payload répond à *"dans quel contexte ?"*. En production, on filtre **avant** de classer : le moteur ne calcule la similarité que sur les points qui passent le filtre, ce qui économise du calcul et **réduit le bruit** (un hit à 0.276 filtré vaut mieux qu'un hit à 0.7 hors-périmètre).

## 7. War-story en miniature : l'index de payload qui éteint le *doom loop*

Voici la reproduction, en réduction, d'un **incident réel** (cf. [04 - Incidents et leçons](docs/04-Incidents-et-Lecons.md), section 8).

En production, l'indexeur déduplique en interrogeant Qdrant par un `scroll` **filtré sur un
champ de hash** (`contentHash`). Tant que ce champ n'a **pas d'index de payload**, chaque
`scroll` filtré est un **balayage `O(n)`** de toute la collection. Sur plus d'un million de
points avec payload sur disque, cela prenait **60 à 95 secondes** par requête — les délais
explosaient, les clients **réessayaient**, et les CPU **saturaient pendant des heures**.

Reproduisons le geste qui a tout réglé : créer l'index de payload. On fabrique une collection
synthétique (vecteurs aléatoires, pas d'embeddings — on ne mesure que le filtre), on chronomètre
un `scroll` filtré **avant** puis **après** la création de l'index.

In [7]:
import time, random, warnings
random.seed(42)

wc = QdrantClient(":memory:")
DIM, N = 16, 20_000
wc.create_collection(
    "hash_demo",
    vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
)
points = [
    models.PointStruct(
        id=i,
        vector=[random.random() for _ in range(DIM)],
        payload={"contentHash": f"h{i % 5000:05d}", "source": "conv" if i % 2 else "code"},
    )
    for i in range(N)
]
wc.upsert("hash_demo", points=points, wait=True)
print(f"{N} points synthétiques upsertés.")

cible = models.Filter(
    must=[models.FieldCondition(key="contentHash", match=models.MatchValue(value="h00042"))]
)

def chrono_scroll(label):
    t0 = time.perf_counter()
    res, _ = wc.scroll("hash_demo", scroll_filter=cible, limit=10, with_payload=False, with_vectors=False)
    dt = (time.perf_counter() - t0) * 1000
    print(f"{label:<18} {dt:7.2f} ms  -> {len(res)} point(s)")
    return dt

t_sans = chrono_scroll("sans index")
# Qdrant local : les index payload sont sans effet (demonstration du concept) - avertissement cible masque
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Payload indexes have no effect", category=UserWarning)
    wc.create_payload_index("hash_demo", field_name="contentHash", field_schema=models.PayloadSchemaType.KEYWORD)
t_avec = chrono_scroll("avec index")

20000 points synthétiques upsertés.


sans index          387.36 ms  -> 4 point(s)


avec index          308.36 ms  -> 4 point(s)


### Lecture du résultat : en mode local, les deux chronos mesurent la même chose

**Mesure brute** : la cellule précédente imprime les deux chronos pour 4 points, et ils sont du même ordre. Ces durées varient d'un passage à l'autre : la sortie ci-dessus reste la source de vérité numérique, et c'est le **sens** de l'écart qui se lit, pas le chiffre au centième. **Ce résultat ne dit rien sur l'index** : Qdrant émet un avertissement explicite, *"Payload indexes have no effect in the local Qdrant"*, car l'optimisation d'index ne s'active **que côté serveur**, pas en mode `:memory:`. Les deux chronos mesurent donc **le même balayage** — l'un avec un index déclaré mais inactif, l'autre sans. Cette expérience ne permet ni d'attribuer l'écart à l'index ou à sa maintenance, ni de prédire ce qui se passera à une autre échelle : pour mesurer l'effet d'un index de payload, il faut un **serveur Qdrant**, et le mesurer là-bas.

**L'asymptote est ce qui compte.** À l'échelle réelle — des **millions** de points, payload **sur disque**, requêtes **concurrentes** — le même balayage `O(n)` passe de quelques millisecondes à **dizaines de secondes**, dépasse le délai d'opération et déclenche la tempête de *retries* (le *scroll doom loop*) qui sature les CPU.

**Les leçons (identiques à l'incident réel) :**

- Un **index de payload n'est pas un luxe** : sans lui, tout filtre est un balayage `O(n)`.
- **Créez les index de payload à la création de la collection** — une collection recréée sans ses index repart sur la même bombe à retardement.
- Un *retry* naïf **amplifie** une panne : remplacer la déduplication par `scroll` par une récupération **par identifiant** (`retrieve()`, `O(1)`) supprime la cause à la racine.

## 8. Note — quantization TurboQuant (`turbo.bits`)

À l'échelle d'un million de vecteurs, l'empreinte mémoire devient le facteur limitant. Qdrant
propose une **quantization** qui comprime les vecteurs. Sur l'infrastructure décrite dans cette
section, on utilise **TurboQuant 4 bits**, configuré par la clé officielle :

```yaml
quantization_config:
  turbo:
    bits: bits4        # valeurs possibles : bits1, bits2, bits4, bits8
    always_ram: true   # garder les vecteurs quantizés en RAM pour des recherches rapides
```

Résultat mesuré en production : **compression 8×**, pour une **score-recall@10 = 1.0** vs la
recherche exacte (aucune perte de qualité observée sur la collection réelle). Le *rescore* est
un paramètre **de requête** (`params.quantization.rescore`), pas de la collection. Détails et
contexte dans [02 - Infrastructure Qdrant](docs/02-Infrastructure-Qdrant.md).

## 9. Pour aller plus loin — la série RAG sur texte

Ce notebook manipule l'**infrastructure** de mémoire sémantique. Pour le versant **applicatif**
du RAG sur des documents, deux notebooks de la section [Texte](../Texte/) sont complémentaires :

- [`5_RAG_Modern.ipynb`](../Texte/05_RAG_Modern.ipynb) — un pipeline RAG complet de bout en bout
  (découpage, embeddings, recherche, génération augmentée).
- [`14_Persistent_Memory.ipynb`](../Texte/14_Persistent_Memory.ipynb) — donner une mémoire
  persistante à un agent conversationnel.

Et pour la théorie : [01 - Pourquoi la mémoire sémantique](docs/01-Pourquoi-Memoire-Semantique.md).

## 10. Exemples guidés et exercices

Cette section combine deux types d'activités. Les **trois exemples guidés**, crédités à la
PR #23248, sont des résolutions complètes et exécutées : ajouter un document et le retrouver
par le sens, combiner deux conditions de filtre, comparer deux modèles d'embeddings. Les
**trois exercices** qui suivent chacun d'eux mesurent autre chose : la même intention posée
dans une autre langue, un filtre d'**exclusion**, et le **coût** (durée d'indexation et
stockage brut des vecteurs) des deux modèles.

Chaque cellule d'exercice est un **squelette exécutable** (elle s'exécute telle quelle, mais ne
fait rien d'utile tant que vous n'avez pas rempli les `# TODO`). Inspirez-vous des sections 4
(recherche sémantique), 6 (filtres de payload), de la section 2 (choix du modèle
d'embeddings) et de l'exemple guidé qui précède chaque exercice.

### Exemple guidé 1 — Retrouver un document par le sens, pas par les mots (crédité #23248)

Le grounding retrouve un texte grâce à sa **signification**, même si la requête ne partage
aucun mot avec lui. C'est tout l'intérêt d'une recherche vectorielle par rapport à un `grep`.

**Objectif.** Ajouter au corpus un document sur la sécurité des secrets (par exemple la
rotation des clés d'API), puis écrire une requête qui le retrouve **sans employer aucun de
ses mots-clés** (« secret », « clé », « API »…).

**Indices.**
- Inspirez-vous de la section 3 (indexation via `models.PointStruct` + `client.upsert(...)`)
  et de la fonction `chercher` (section 4) pour la requête.
- Un modèle multilingue rapproche par exemple « gestion des identifiants » et « sécurité des
  secrets » bien qu'aucun mot ne soit commun.

> **Exemple guidé (crédité #23248).** La résolution ci-dessous est celle du groupe de la
> PR #23248 (TP MSAPP3IN08), conservée telle quelle : elle ajoute le document sous un
> identifiant fixe — relancer la cellule remplace le point au lieu de créer un doublon —,
> vérifie qu'aucun mot plein n'est commun entre le document et la requête, puis mesure le
> rang obtenu. L'exercice 1bis qui suit pose la même intention dans une autre langue.

In [8]:
# Exercice 1 — Ajoutez un document sur la sécurité des secrets, puis retrouvez-le
# par le SENS, sans citer ses mots exacts (cf. section 4 : fonction `chercher`).
import re

nouveau_document = (
    "Les clés d'API sont renouvelées chaque mois et ne doivent jamais "
    "apparaître en clair dans un dépôt de code."
)
metadata_secrets = {"source": "exercice", "sujet": "secrets"}
requete_semantique = "comment protéger les identifiants d'accès à un service tiers ?"

# Identifiant fixe, à la suite de ceux de la cellule 6 : relancer la cellule
# remplace le point au lieu de créer un doublon.
client.upsert(
    collection_name="grounding_demo",
    points=[
        models.PointStruct(
            id=len(documents),
            vector=models.Document(text=nouveau_document, model=EMBED_MODEL),
            payload={**metadata_secrets, "texte": nouveau_document},
        )
    ],
)


def mots_pleins(texte: str) -> set[str]:
    """Mots de plus de trois lettres, en minuscules (les mots outils sont ignorés)."""
    return {m for m in re.findall(r"\w+", texte.lower()) if len(m) > 3}


# Vérification lexicale : la requête ne doit partager aucun mot plein avec le document.
mots_communs = mots_pleins(nouveau_document) & mots_pleins(requete_semantique)
print("Mots pleins communs document/requête :", mots_communs or "aucun")
print()

resultat = chercher(requete_semantique, limit=3)
documents_trouves = [h.payload["texte"] for h in resultat]
if nouveau_document in documents_trouves:
    rang = documents_trouves.index(nouveau_document) + 1
    print(f"Document sur les secrets retrouvé au rang {rang} du top 3.")
else:
    print("Document sur les secrets absent du top 3 : reformulez la requête.")

Mots pleins communs document/requête : aucun

Requête : "comment protéger les identifiants d'accès à un service tiers ?"

  score=0.544  [code/rate-limiting]  Le throttling des appels sortants protège le service des pics de charge.
  score=0.403  [conversation/rate-limiting]  Pour éviter de saturer l'API, on limite le nombre de requêtes par seconde.
  score=0.175  [exercice/secrets]  Les clés d'API sont renouvelées chaque mois et ne doivent jamais apparaître en clair dans un dépôt de code.

Document sur les secrets retrouvé au rang 3 du top 3.


### Lecture du résultat : retrouvé par le sens, mais au rang 3

La vérification lexicale affiche *"Mots pleins communs document/requête : aucun"* : la requête *"comment protéger les identifiants d'accès à un service tiers ?"* ne reprend aucun mot de plus de trois lettres du document (« clés », « renouvelées », « dépôt », « code »...). Le sigle « API », trop court pour ce contrôle, n'y figure pas non plus. Seuls des mots outils comme « les » ou « un » sont partagés. Le document sur les secrets remonte pourtant dans le top 3, ce qui remplit l'objectif : un `grep` sur les mots pleins de la requête ne l'aurait pas trouvé.

Le classement invite cependant à la prudence. Le document n'arrive qu'au **rang 3**, avec un score de **0.175**, loin derrière les deux fragments *rate-limiting* (**0.544** et **0.403**). La requête parle de « service », un mot présent dans le document sur le throttling : une hypothèse, non mesurée ici, est que ce recouvrement de vocabulaire avec un **autre** document suffit à le faire passer devant. Avec `limit=2`, l'exercice aurait échoué. Sur cette même requête, l'exemple guidé 3 montre que le modèle `bge-small-en` classe le document au rang 1 : le rang obtenu dépend aussi du modèle, pas seulement de la formulation.

### Exercice 1bis — La même intention, dans une autre langue

L'exemple guidé 1 a retrouvé le document sur les secrets avec une requête **française**, au
rang 3. Le modèle d'embeddings est dit **multilingue** : cela suffit-il à retrouver le même
document depuis une requête **anglaise** ?

**Objectif.** Posez la même intention en anglais sur la même collection, mesurez le rang du
document dont le sujet est `secrets`, et comparez-le au rang obtenu en français.

**Indices.**
- `# Étape 1` : écrivez `requete_anglaise`, une phrase anglaise exprimant « protéger les
  identifiants d'accès », sans reprendre les mots du document.
- `# Étape 2` : interrogez `grounding_demo` en demandant **tout** le classement (un `limit`
  égal au nombre de points de la collection), puis relevez `rang_anglais`, la position du
  document dont `sujet == "secrets"` (1 pour le premier).
- `# Étape 3` : écrivez `verdict_langue`, une phrase qui compare `rang_anglais` au rang 3
  obtenu en français : meilleur, identique ou dégradé.
- `# Indice` : `[h.metadata["sujet"] for h in hits].index("secrets") + 1` donne le rang ;
  aucun appel réseau n'est nécessaire.

In [9]:
# Exercice 1bis — la même intention posée dans une autre langue
# TODO étudiant : mesurer le rang du document "secrets" depuis une requête ANGLAISE
#   Étape 1 : écrire `requete_anglaise` (même intention, sans les mots du document)
#   Étape 2 : interroger toute la collection, relever `rang_anglais` (1 pour le premier)
#   Étape 3 : comparer au rang 3 obtenu en français et écrire `verdict_langue`
requete_anglaise = "..."  # TODO étudiant : votre formulation en anglais
rang_anglais = None  # TODO étudiant
verdict_langue = None  # TODO étudiant
print("Exercice 1bis à compléter — rang du document secrets depuis une requête anglaise :",
      rang_anglais)

Exercice 1bis à compléter — rang du document secrets depuis une requête anglaise : None


### Exemple guidé 2 — Combiner deux conditions de filtre (ET logique) (crédité #23248)

Un `Filter(must=[...])` ne retient que les points qui satisfont **toutes** les
`FieldCondition` listées : c'est l'équivalent d'un ET logique. La section 6 filtre sur une
seule condition ; ici on en combine deux.

**Objectif.** Construire un filtre qui sélectionne les documents dont `source = "incident"`
**et** `sujet = "mount-drift"`, puis interroger la collection avec ce filtre.

**Indices.**
- Inspectez d'abord les métadonnées de la cellule 6 pour connaître les valeurs exactes.
- Deux `FieldCondition` dans la même liste `must` = intersection (ET). Pour un OU,
  il faudrait `should=[...]`.

> **Exemple guidé (crédité #23248).** La résolution ci-dessous est celle du groupe de la
> PR #23248 (TP MSAPP3IN08), conservée telle quelle. Elle ajoute une **contre-épreuve** :
> la même requête avec la seule condition `source` montre que la seconde condition retire
> exactement le point attendu. L'exercice 2bis qui suit passe du ET au **NON**.

In [10]:
# Exercice 2 — Combinez DEUX conditions de filtre (ET logique via `must`).
# Cf. section 6 : un `Filter(must=[...])` ne retient que les points satisfaisant
# TOUTES les FieldCondition listées.

filtre_combine = models.Filter(
    must=[
        models.FieldCondition(key="source", match=models.MatchValue(value="incident")),
        models.FieldCondition(key="sujet", match=models.MatchValue(value="mount-drift")),
    ]
)

reponse = client.query_points(
    collection_name="grounding_demo",
    query=models.Document(text="dérive de montage du disque", model=EMBED_MODEL),
    query_filter=filtre_combine,
    limit=5,
)
print(f"Hits avec le filtre source=incident ET sujet=mount-drift : {len(reponse.points)}")
for h in reponse.points:
    print(
        f"  score={h.score:.3f}  "
        f"[{h.payload['source']}/{h.payload['sujet']}]  {h.payload['texte']}"
    )

# Contre-épreuve : la seule condition source=incident laisse passer plus de points.
filtre_source_seule = models.Filter(must=[filtre_combine.must[0]])
reponse_source = client.query_points(
    collection_name="grounding_demo",
    query=models.Document(text="dérive de montage du disque", model=EMBED_MODEL),
    query_filter=filtre_source_seule,
    limit=5,
)
print(f"\nHits avec la seule condition source=incident : {len(reponse_source.points)}")
for h in reponse_source.points:
    print(
        f"  score={h.score:.3f}  "
        f"[{h.payload['source']}/{h.payload['sujet']}]  {h.payload['texte']}"
    )

Hits avec le filtre source=incident ET sujet=mount-drift : 1
  score=0.399  [incident/mount-drift]  La collection a été recréée après une dérive de montage du disque virtuel.

Hits avec la seule condition source=incident : 2
  score=0.399  [incident/mount-drift]  La collection a été recréée après une dérive de montage du disque virtuel.
  score=0.201  [incident/payload-index]  Un index de payload sur le champ de hash rend la déduplication instantanée.


### Lecture du résultat : le ET logique réduit le périmètre à un seul point

Avec les deux conditions dans `must`, la requête renvoie **1 hit** : le fragment *[incident/mount-drift]*, score **0.399**. La contre-épreuve avec la seule condition `source = "incident"` en renvoie **2** : le même fragment (**0.399**) et *[incident/payload-index]* (**0.201**). La seconde `FieldCondition` a donc bien retiré le point qui avait la bonne source mais pas le bon sujet : c'est l'intersection attendue.

Le score du fragment *mount-drift* est identique dans les deux cas (**0.399**) : le filtre ne modifie pas la similarité, il restreint seulement l'ensemble des points renvoyés. Le `limit=5` n'est pas atteint : quand le filtre ne laisse passer qu'un point, la requête ne renvoie qu'un point, quel que soit le `limit`.

### Exercice 2bis — Du ET au NON : exclure une source

L'exemple guidé 2 a construit l'**intersection** de deux conditions (`must`). Le complément
existe : `must_not` retire les points qui satisfont une condition et laisse tous les autres.

**Objectif.** Construire un filtre qui **exclut** les fragments de `source = "incident"`,
interroger la collection avec ce filtre, et comparer le nombre de hits à celui de la requête
sans filtre.

**Indices.**
- `# Étape 1` : construisez `filtre_exclusion` avec `models.Filter(must_not=[...])` et **une**
  `FieldCondition` sur `key="source"`.
- `# Étape 2` : interrogez avec `query_filter=filtre_exclusion` en demandant tout le
  classement, et relevez `nb_hits_exclus`.
- `# Étape 3` : calculez `nb_points_exclus`, le nombre de points que l'exclusion a retirés
  (`client.count("grounding_demo").count` moins `nb_hits_exclus`), et vérifiez que les deux
  fragments `incident` de l'exemple guidé 2 en font bien partie.
- `# Indice` : `must_not` est le complément de `must` ; un point exclu ne peut pas réapparaître
  dans le classement, quel que soit `limit`.

In [11]:
# Exercice 2bis — exclure une source (le complément du ET logique)
# TODO étudiant : construire un filtre `must_not` et compter ce qu'il retire
#   Étape 1 : `filtre_exclusion` = un filtre avec `must_not` et une seule condition sur `source`
#   Étape 2 : interroger toute la collection avec `client.query_points(...)` et ce filtre
#             -> `nb_hits_exclus`
#   Étape 3 : `nb_points_exclus` = total - `nb_hits_exclus`, et vérifier que les incidents partent
filtre_exclusion = None  # TODO étudiant
nb_hits_exclus = None  # TODO étudiant
nb_points_exclus = None  # TODO étudiant
print("Exercice 2bis à compléter — hits restants après exclusion :", nb_hits_exclus)

Exercice 2bis à compléter — hits restants après exclusion : None


### Exemple guidé 3 — Comparer deux modèles d'embeddings (crédité #23248)

Le choix du modèle d'embeddings (langue couverte, dimension du vecteur) change la qualité du
retrieval. `fastembed` supporte plusieurs modèles locaux ; la section 2 en a fixé un
(multilingue, 384 dimensions).

**Objectif.** Créer un **second client** avec un modèle d'embeddings **différent**
(par exemple `BAAI/bge-small-en-v1.5`), ré-indexer le même petit corpus, puis comparer ce que
les deux modèles renvoient pour une même requête.

**Indices.**
- Créez un nouveau `QdrantClient(":memory:")`, ouvrez votre collection avec
  `create_collection(...)` (même dimension, même distance), puis déclarez le modèle sur
  chaque `models.Document(text=..., model=...)`.
- Un modèle entraîné surtout sur l'anglais donnera des scores plus faibles sur un corpus
  francophone : c'est précisément ce que l'on veut mettre en évidence.
- Pour lister les modèles disponibles : la doc `fastembed`
  (`fastembed.TextEmbedding.list_supported_models()`).

> **Exemple guidé (crédité #23248).** La résolution ci-dessous est celle du groupe de la
> PR #23248 (TP MSAPP3IN08), conservée telle quelle. Elle ne compare pas les **scores** —
> chaque modèle a sa propre échelle — mais le **rang du premier document pertinent**, et elle
> mesure cette échelle sur toutes les paires requête/document. L'exercice 3bis qui suit
> abandonne la qualité pour mesurer le **coût** des deux mêmes modèles.

In [12]:
# Exercice 3 — Changez de modèle d'embeddings et comparez les scores.
# Cf. cellule 4 : le modèle utilisé est celui déclaré dans chaque `models.Document`.

MODELE_B = "BAAI/bge-small-en-v1.5"
client_b = QdrantClient(":memory:")
client_b.create_collection(
    "comparaison",
    vectors_config=models.VectorParams(size=EMBED_DIM, distance=models.Distance.COSINE),
)

# Même corpus que "grounding_demo" : les 6 documents de la cellule 6 + celui de l'exercice 1.
corpus = documents + [nouveau_document]
corpus_meta = metadata + [metadata_secrets]
points_b = [
    models.PointStruct(
        id=i,
        vector=models.Document(text=texte, model=MODELE_B),
        payload={**meta, "texte": texte},
    )
    for i, (texte, meta) in enumerate(zip(corpus, corpus_meta))
]
client_b.upsert(collection_name="comparaison", points=points_b)

clients = {"multilingue": (client, "grounding_demo"), "bge-small-en": (client_b, "comparaison")}
for nom, (cl, collection) in clients.items():
    dimension = cl.get_collection(collection).config.params.vectors.size
    print(f"{nom:<13} points={cl.count(collection).count}  dimension={dimension}")

# Requêtes déjà utilisées dans le notebook, avec le sujet du document attendu.
requetes = {
    "comment ne pas dépasser le quota d'appels ?": "rate-limiting",
    "rate limiting": "rate-limiting",
    "éviter les pics de charge sur le service": "rate-limiting",
    "ralentir les appels réseau": "rate-limiting",
    "dérive de montage du disque": "mount-drift",
    requete_semantique: "secrets",
}

# Les scores ne se comparent pas d'un modèle à l'autre (échelles différentes) :
# on compare le RANG du premier document pertinent dans chaque classement.
comparaison = {nom: {} for nom in clients}
tous_les_scores = {nom: [] for nom in clients}
for question, sujet_attendu in requetes.items():
    print(f"\nRequête : {question!r}  (sujet attendu : {sujet_attendu})")
    for nom, (cl, collection) in clients.items():
        modele = EMBED_MODEL if nom == "multilingue" else MODELE_B
        reponse = cl.query_points(
            collection_name=collection,
            query=models.Document(text=question, model=modele),
            limit=len(corpus),
        )
        hits = reponse.points
        tous_les_scores[nom] += [h.score for h in hits]
        sujets = [h.payload["sujet"] for h in hits]
        rang = sujets.index(sujet_attendu) + 1
        comparaison[nom][question] = rang
        top3 = ", ".join(
            f"{h.payload['source']}/{h.payload['sujet']} ({h.score:.3f})" for h in hits[:3]
        )
        print(f"  {nom:<13} rang du pertinent = {rang}   top 3 : {top3}")

print("\nRésumé — rang du premier document pertinent (1 = idéal) :")
for nom, rangs in comparaison.items():
    premiers = sum(r == 1 for r in rangs.values())
    print(f"  {nom:<13} rang 1 sur {premiers}/{len(rangs)} requêtes, rang moyen = {sum(rangs.values()) / len(rangs):.2f}")

SEUIL = 0.3  # seuil de pertinence suggéré en section 4
print(f"\nÉchelle des scores sur toutes les paires requête/document (seuil {SEUIL}) :")
for nom, scores in tous_les_scores.items():
    au_dessus = sum(s >= SEUIL for s in scores)
    print(f"  {nom:<13} min={min(scores):.3f}  max={max(scores):.3f}  paires >= seuil : {au_dessus}/{len(scores)}")

multilingue   points=7  dimension=384
bge-small-en  points=7  dimension=384

Requête : "comment ne pas dépasser le quota d'appels ?"  (sujet attendu : rate-limiting)
  multilingue   rang du pertinent = 1   top 3 : code/rate-limiting (0.667), conversation/rate-limiting (0.519), incident/payload-index (0.113)
  bge-small-en  rang du pertinent = 1   top 3 : code/rate-limiting (0.702), bruit/hors-domaine (0.677), exercice/secrets (0.673)

Requête : 'rate limiting'  (sujet attendu : rate-limiting)
  multilingue   rang du pertinent = 1   top 3 : code/rate-limiting (0.349), conversation/rate-limiting (0.344), doc/quantization (0.317)


  bge-small-en  rang du pertinent = 1   top 3 : code/rate-limiting (0.665), conversation/rate-limiting (0.619), doc/quantization (0.596)

Requête : 'éviter les pics de charge sur le service'  (sujet attendu : rate-limiting)
  multilingue   rang du pertinent = 1   top 3 : code/rate-limiting (0.714), conversation/rate-limiting (0.390), incident/payload-index (0.207)
  bge-small-en  rang du pertinent = 1   top 3 : code/rate-limiting (0.844), incident/mount-drift (0.709), bruit/hors-domaine (0.683)

Requête : 'ralentir les appels réseau'  (sujet attendu : rate-limiting)
  multilingue   rang du pertinent = 1   top 3 : code/rate-limiting (0.615), conversation/rate-limiting (0.428), incident/payload-index (0.158)


  bge-small-en  rang du pertinent = 3   top 3 : incident/mount-drift (0.703), bruit/hors-domaine (0.686), code/rate-limiting (0.667)

Requête : 'dérive de montage du disque'  (sujet attendu : mount-drift)
  multilingue   rang du pertinent = 1   top 3 : incident/mount-drift (0.399), code/rate-limiting (0.260), incident/payload-index (0.201)
  bge-small-en  rang du pertinent = 1   top 3 : incident/mount-drift (0.835), incident/payload-index (0.657), bruit/hors-domaine (0.643)

Requête : "comment protéger les identifiants d'accès à un service tiers ?"  (sujet attendu : secrets)
  multilingue   rang du pertinent = 3   top 3 : code/rate-limiting (0.544), conversation/rate-limiting (0.403), exercice/secrets (0.175)


  bge-small-en  rang du pertinent = 1   top 3 : exercice/secrets (0.703), code/rate-limiting (0.674), conversation/rate-limiting (0.651)

Résumé — rang du premier document pertinent (1 = idéal) :
  multilingue   rang 1 sur 5/6 requêtes, rang moyen = 1.33
  bge-small-en  rang 1 sur 5/6 requêtes, rang moyen = 1.33

Échelle des scores sur toutes les paires requête/document (seuil 0.3) :
  multilingue   min=-0.120  max=0.714  paires >= seuil : 12/42
  bge-small-en  min=0.452  max=0.844  paires >= seuil : 42/42


### Lecture du résultat : égalité sur les classements, échelles de score incomparables

Les deux collections contiennent les mêmes **7 points** et produisent des vecteurs de même dimension (**384**) : seule la manière d'embarquer le texte change. On compare donc le **rang du premier document pertinent**, pas les scores.

| Requête | multilingue | bge-small-en |
|---|---|---|
| *comment ne pas dépasser le quota d'appels ?* | 1 | 1 |
| *rate limiting* | 1 | 1 |
| *éviter les pics de charge sur le service* | 1 | 1 |
| *ralentir les appels réseau* | 1 | **3** |
| *dérive de montage du disque* | 1 | 1 |
| *comment protéger les identifiants d'accès à un service tiers ?* | **3** | 1 |

Sur ces 6 requêtes, chaque modèle place le document pertinent au rang 1 dans **5 cas sur 6** (rang moyen **1.33** pour les deux). Les deux modèles ne se trompent pas sur la même requête. Sur *"ralentir les appels réseau"*, `bge-small-en` place *incident/mount-drift* et la tarte aux pommes (*bruit/hors-domaine*) devant le fragment *code/rate-limiting*. Le modèle multilingue, lui, classe le document sur les secrets au rang 3. Ce jeu de test a deux limites : les requêtes ont été écrites (sections 4 et 5) ou validées (exemple guidé 1) avec le modèle multilingue, et le corpus comme 5 requêtes sur 6 sont en français alors que `bge-small-en` est un modèle anglais. L'égalité **5/6** ne vaut que pour ce jeu de 6 requêtes et 7 documents : elle ne permet pas de désigner un meilleur modèle.

L'indice de l'énoncé suppose que les niveaux de score se comparent d'un modèle à l'autre. Ce n'est pas le cas : chaque modèle utilise sa propre plage. Chez `bge-small-en`, tous les scores tiennent entre **0.452** et **0.844**, si bien que le témoin négatif (la tarte aux pommes, **0.677**) arrive au rang 2 sur la requête *quota d'appels*. Chez le modèle multilingue, il ne figure dans aucun top 3. La dernière mesure le montre sur les 42 paires requête/document : le seuil `score ≥ 0.3` suggéré en section 4 en retient **12/42** avec le modèle multilingue (scores de **-0.120** à **0.714**), mais **42/42** avec `bge-small-en`, pour qui il ne filtre plus rien. Un score n'a de sens qu'à l'intérieur d'un modèle : seul le classement se compare, et un seuil se recalibre à chaque changement de modèle.

### Exercice 3bis — Ce que coûte un modèle : durée d'indexation et stockage des vecteurs

L'exemple guidé 3 a mesuré la **qualité** des deux modèles : rang 1 sur cinq des six requêtes
pour chacun, mais pas sur les mêmes. La qualité ne départage donc pas ; reste le **coût**.

**Objectif.** Mesurer, pour les deux modèles déjà construits, la durée d'indexation
**bout-en-bout** du corpus et la taille de **stockage brut** des vecteurs, puis conclure sur un
critère chiffré.

**Indices.**
- `# Étape 1` : pour chaque modèle, créez un client neuf (`QdrantClient(":memory:")`, puis
  `create_collection(...)`) et chronométrez la ré-indexation complète des mêmes documents avec
  `time.perf_counter()` — la durée mesurée est **bout-en-bout** : chargement du modèle,
  encodage des textes et insertion des points ; rangez les durées dans `temps_indexation`.
- `# Étape 2` : calculez le **stockage brut théorique** des vecteurs —
  `nombre_de_points × dimension × 4` octets, un `float32` par composante — et rangez-le dans
  `stockage_octets`. Ce compte exclut le modèle en RAM, les structures d'index, le payload et
  l'overhead du moteur : ce n'est pas la RAM mesurée du processus.
- `# Étape 3` : écrivez `choix_motivation`, une phrase qui nomme le chiffre qui départage les
  deux modèles **sur votre mesure** — avec sa limite : sept documents et un seul passage mesuré
  ne s'extrapolent pas à un corpus d'un million de fragments.
- `# Indice` : la dimension se lit dans `cl.get_collection(nom).config.params.vectors.size`,
  comme dans l'exemple guidé 3 ; le premier passage inclut le chargement du modèle — signalez-le
  dans votre conclusion plutôt que de le masquer.

In [13]:
# Exercice 3bis — le coût des deux modèles : durée d'indexation et stockage des vecteurs
# TODO étudiant : chronométrer l'indexation bout-en-bout et chiffrer le stockage brut, puis trancher sur un chiffre
#   Étape 1 : ré-indexer le même corpus sur un client neuf par modèle -> `temps_indexation` (chargement + encodage + insertion)
#   Étape 2 : `stockage_octets` = nombre de points × dimension × 4 (float32 brut, hors modèle/index/payload)
#   Étape 3 : `choix_motivation` (le chiffre qui départage, et sa limite)
import time

temps_indexation = None  # TODO étudiant : {nom_modele: secondes}
stockage_octets = None  # TODO étudiant : {nom_modele: octets}
choix_motivation = None  # TODO étudiant
print("Exercice 3bis à compléter — durées d'indexation mesurées :", temps_indexation)

Exercice 3bis à compléter — durées d'indexation mesurées : None


## 11. Pour aller plus loin — explorations libres

Deux pistes **ouvertes** (sans squelette de code). Les activités « ajouter un document »,
« combiner deux conditions de filtre » et « changer de modèle d'embeddings » ne sont plus
listées ici : la section 10 les traite, en exemple guidé puis en exercice.

1. **Faites varier `N`** dans la section 7 (par ex. 5 000 → 100 000) et observez comment l'écart
   de temps `scroll` avec/sans index évolue.
2. **Passez du mode mémoire au disque** : instanciez `QdrantClient(path=...)` — ou une instance
   Docker — et rejouez la section 3. Qu'est-ce qui survit à l'arrêt du processus, et qu'est-ce
   qui change dans les temps d'écriture ?

## Continuer le parcours

Vous avez maintenant relié texte, embeddings, recherche sémantique et filtres Qdrant. La suite immédiate est [`02-Retrieval-Avance.ipynb`](02-Retrieval-Avance.ipynb) : elle part de ces résultats pour travailler le classement, la fusion et l'évaluation du retrieval.

***

*Notebook pratique — Section RAG et Mémoire Sémantique — Juin 2026. Exécutable hors ligne
(Qdrant en mémoire + fastembed), sans clé d'API ni donnée réelle.*